# Fine-tune Depth Anything V2 - camera learns metric 3D (Direction B)

Teaches the monocular depth model metric depth for your fixed camera, supervised by sparse LiDAR depth. **scale-align only** (lstsq, no training) or **LoRA fine-tune** with a masked loss. Real pairs come from the calibrated harvest; `USE_SAMPLE_DATA` runs it now.

Config split into **SWITCHES / QUANTITATIVE / QUALITATIVE**.

## 1. Config

In [ ]:
# SWITCHES (on/off)
USE_GPU          = True
SCALE_ALIGN_ONLY = True
USE_LORA         = True
FREEZE_ENCODER   = True
USE_SAMPLE_DATA  = True
MASK_INVALID     = True

In [ ]:
# QUANTITATIVE (numbers / knobs)
EPOCHS        = 10
BATCH         = 2
LR            = 1e-4
WEIGHT_DECAY  = 0.01
IMG_SIZE      = 518
MIN_DEPTH_M   = 0.3
MAX_DEPTH_M   = 12.0
LORA_RANK     = 8
LORA_ALPHA    = 16
DEVICE_ID     = 0

In [ ]:
# QUALITATIVE (strings / choices)
BASE_MODEL = 'depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf'
DATA_DIR   = 'datasets/depth_pairs'   # images/*.png + depth/*.npy (sparse metres, 0 = none)
OUTPUT_DIR = 'runs/depth_finetune'
LOSS       = 'silog'                  # 'silog' or 'l1'
LORA_TARGETS = ['query', 'value']

## 2. Data (real harvested pairs, or synthetic placeholders)

In [ ]:
import os, glob, numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class DepthPairs(Dataset):
    def __init__(self, root):
        self.root = root; self.imgs = sorted(glob.glob(os.path.join(root, 'images', '*')))
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img = np.asarray(Image.open(self.imgs[i]).convert('RGB'))
        base = os.path.splitext(os.path.basename(self.imgs[i]))[0]
        dp = os.path.join(self.root, 'depth', base + '.npy')
        d = np.load(dp).astype('float32') if os.path.exists(dp) else np.zeros(img.shape[:2], 'float32')
        return img, d

def make_sample(root, n=4, hw=(360, 640)):
    os.makedirs(root + '/images', exist_ok=True); os.makedirs(root + '/depth', exist_ok=True)
    for k in range(n):
        Image.fromarray((np.random.rand(*hw, 3) * 255).astype('uint8')).save(f'{root}/images/s{k}.png')
        d = np.zeros(hw, 'float32'); m = np.random.rand(*hw) < 0.05
        d[m] = np.random.uniform(MIN_DEPTH_M, MAX_DEPTH_M, int(m.sum())); np.save(f'{root}/depth/s{k}.npy', d)

if USE_SAMPLE_DATA and not glob.glob(DATA_DIR + '/images/*'):
    make_sample(DATA_DIR); print('wrote placeholder pairs to', DATA_DIR)
ds = DepthPairs(DATA_DIR); print(len(ds), 'pairs')

## 3a. Scale-align only (no training)

In [ ]:
import numpy as np
from PIL import Image
from transformers import pipeline
if SCALE_ALIGN_ONLY and len(ds):
    pipe = pipeline('depth-estimation', model=BASE_MODEL, device=(DEVICE_ID if USE_GPU else -1))
    img, depth = ds[0]
    pred = pipe(Image.fromarray(img))['predicted_depth'].squeeze().cpu().numpy()
    pred = np.asarray(Image.fromarray(pred).resize((depth.shape[1], depth.shape[0])))
    m = depth > 0
    if m.sum() > 1:
        a, b = np.linalg.lstsq(np.stack([pred[m], np.ones(int(m.sum()))], 1), depth[m], rcond=None)[0]
        print(f'metric_depth = {a:.4f} * pred + {b:.4f}  (apply to the full map)')

## 3b. LoRA fine-tune (masked loss on LiDAR pixels) - `pip install peft`

In [ ]:
import torch, numpy as np
if not SCALE_ALIGN_ONLY:
    from transformers import AutoModelForDepthEstimation, AutoImageProcessor
    proc = AutoImageProcessor.from_pretrained(BASE_MODEL)
    model = AutoModelForDepthEstimation.from_pretrained(BASE_MODEL)
    dev = torch.device('cuda' if (USE_GPU and torch.cuda.is_available()) else 'cpu')
    if FREEZE_ENCODER:
        for n, p in model.named_parameters():
            if 'backbone' in n or 'encoder' in n: p.requires_grad_(False)
    if USE_LORA:
        from peft import LoraConfig, get_peft_model
        model = get_peft_model(model, LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGETS))
    model.to(dev).train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
    def loss_fn(pred, gt):
        m = (gt > MIN_DEPTH_M) & (gt < MAX_DEPTH_M) if MASK_INVALID else torch.ones_like(gt, dtype=torch.bool)
        if m.sum() == 0: return pred.sum() * 0
        if LOSS == 'l1': return (pred[m] - gt[m]).abs().mean()
        d = torch.log(pred[m].clamp(min=1e-3)) - torch.log(gt[m].clamp(min=1e-3))
        return torch.sqrt((d**2).mean() - 0.85 * d.mean()**2)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=lambda b: b)
    for ep in range(EPOCHS):
        tot = 0.0
        for batch in dl:
            opt.zero_grad(); loss = 0.0
            for img, depth in batch:
                out = model(**proc(images=img, return_tensors='pt').to(dev)).predicted_depth.squeeze()
                gt = torch.tensor(depth, device=dev)
                out = torch.nn.functional.interpolate(out[None, None], size=gt.shape, mode='bilinear')[0, 0]
                loss = loss + loss_fn(out, gt)
            loss = loss / len(batch); loss.backward(); opt.step(); tot += float(loss)
        print(f'epoch {ep+1}/{EPOCHS} loss {tot/len(dl):.4f}')
    import os; os.makedirs(OUTPUT_DIR, exist_ok=True); model.save_pretrained(OUTPUT_DIR); print('saved', OUTPUT_DIR)